# Data Science u kulturi — praktične vježbe (AI-first)
### Colab uz kolegij | ak. god. 2026./2027.

**Nositelj:** izv. prof. dr. sc. Benedikt Perak

Ovaj Colab prati cijeli data-science pipeline u kulturološkim istraživanjima,
s naglaskom na **suvremene AI alate**: velike jezične modele (LLM), embedding
reprezentacije, semantičku pretragu, RAG (retrieval-augmented generation) i
agentske sustave.

**Temeljni resurs:**
📖 **Perak, B. (2025). *Komunikacija u doba umjetne inteligencije: Razvoj velikih jezičnih modela i komunikacijskih agenata*. Rijeka: Filozofski fakultet u Rijeci.**
Open access: [GitHub bperak/komunikacija_u_doba_ai](https://github.com/bperak/komunikacija_u_doba_ai) | ISBN 978-953-361-147-1

---

## Sadržaj
1. AI revolucija u humanistici: od tablica do agenata
2. Podaci u kulturi: FAIR principi i AI-spremni podaci
3. Pandas + LLM: razgovor s podacima
4. Embedding: semantička pretraga kulturnih zbirki
5. RAG: razgovor s kulturnom baštinom
6. Agentski sustavi: automatizirani istraživač
7. Etika i kritičko vrednovanje AI rezultata
8. Vježbe 🟢🟡🏆

---
## 0. Postavljanje

Instaliramo biblioteke. **Ključ:** [aistudio.google.com](https://aistudio.google.com) → API key (besplatno, Gemini Flash free tier).

In [ ]:
# @title Instalacija i setup
!pip install -q pandas numpy matplotlib seaborn scipy scikit-learn google-generativeai

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
sns.set_palette("husl")

import os
from google.colab import userdata
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    os.environ['GEMINI_API_KEY'] = input("Gemini API ključ: ")

import google.generativeai as genai
genai.configure(api_key=os.environ['GEMINI_API_KEY'])
MODEL = "gemini-2.0-flash"
model = genai.GenerativeModel(MODEL)

print("✅ Spremno:", pd.__version__)

---
## 1. AI revolucija u humanistici

Data science u kulturi više nije samo statistika i tablice. Suvremeni istraživač
koristi **cijeli AI stack**:

| Tradicionalno | Suvremeno (AI-first) |
|---------------|---------------------|
| Ručno kodiranje varijabli | Automatsko označavanje LLM-om |
| Pretraga po ključnim riječima | Semantička pretraga (embedding) |
| Statički izvještaj | Interaktivni RAG asistent |
| Analiza odvojeno od interpretacije | Agent koji analizira I interpretira |
| Jedan format podataka | Multimodalno (tekst, slika, govor) |

**Dvije ključne ideje iz knjige *Komunikacija u doba umjetne inteligencije*:**
1. LLM-ovi su **komunikacijski agenti** — ne samo alati za tekst, nego sučelja prema podacima
2. Kritičko vrednovanje je obavezno: modeli **haluciniraju**, pristranosti se prenose iz korpusa

---
## 2. Podaci u kulturi: FAIR principi

Kulturni podaci (katalozi, metapodaci, digitalizirana baština) moraju biti:

- **F**indable (pronalazivi) — metapodaci, perzistentni identifikatori
- **A**ccessible (dostupni) — otvoreni API-ji
- **I**nteroperable (interoperabilni) — standardni formati (CSV, JSON-LD)
- **R**eusable (ponovno upotrebljivi) — licence, dokumentacija

**AI-spremni podaci** = čisti, označeni, dokumentirani podaci koje LLM/agent
može direktno koristiti. Ovo je danas ključna vještina digitalne humanistike.

In [ ]:
# @title FAIR primjer: metapodaci muzejske zbirke
# Primjer: digitalizirani katalog (sintetički podaci po uzoru na javne zbirke)
katalog = pd.DataFrame({
    "id": ["M-0001", "M-0002", "M-0003", "M-0004", "M-0005"],
    "naziv": ["Portret ribara", "Mrtva priroda s cvijećem", "Pejzaž Kvarnera",
              "Apstraktna kompozicija", "Stari Rijeka"],
    "autor": ["Kralj", "Vidović", "Matoš", "Šebalj", "nepoznat"],
    "godina": [1920, 1931, 1915, 1958, 1900],
    "materijal": ["ulje/platno", "ulje/platno", "akvarel", "ulje/platno", "fotografija"],
    "dimenzije_cm": [80, 65, 45, 100, 30],
    "zbirka": ["Moderna", "Moderna", "Grafika", "Suvremena", "Fotografija"],
})

print("=== Katalog (AI-spremni tablični podaci) ===")
print(katalog)
print()

# FAIR: provjera kvalitete
print("=== Provjera kvalitete (FAIR readiness) ===")
print("  Null vrijednosti po stupcu:")
print(katalog.isnull().sum())
print(f"  Duplikati: {katalog.duplicated().sum()}")
print(f"  Nepotpuni zapisi (nepoznat autor): {(katalog['autor']=='nepoznat').sum()}")

# Priprema za AI: metapodaci kao JSON struktura
import json
ai_ready = katalog.to_dict(orient='records')
print(f"\n=== AI-ready (JSON, {len(ai_ready)} zapisa) ===")
print(json.dumps(ai_ready[:2], ensure_ascii=False, indent=2))

---
## 3. Pandas + LLM: razgovor s podacima

Umiesto ručnog pisanja Pandas upita, LLM može **generirati analitički kod**
iz prirodnojezičnog pitanja. Ovo je "chat s podacima" — pristup koji knjiga
opisuje kao **komunikacijskog agenta za podatke**.

In [ ]:
# @title LLM generira analizu podataka iz pitanja
# Anketa o kulturnim navikama
anketa = pd.DataFrame({
    "dob": np.random.default_rng(42).integers(18, 30, 40),
    "kino": np.random.default_rng(1).integers(0, 25, 40),
    "citam": np.random.default_rng(2).integers(0, 12, 40),
    "koncerti": np.random.default_rng(3).integers(0, 12, 40),
})
anketa["kultura"] = anketa["kino"] + anketa["koncerti"] * 2

pitanje = "Koja je prosječna vrijednost kultura indeksa za studente koji citaju vise od 5 sati tjedno?"

prompt = f"""
Pandas DataFrame 'anketa' ima stupce: dob, kino, citam, koncerti, kultura.
Napiši SAMO Python kod (bez objašnjenja) koji odgovara na pitanje:

Pitanje: {pitanje}

Vrati kod u formatu koji se može direktno izvršiti (bez markdowna).
"""

odgovor = model.generate_content(prompt)
kod = odgovor.text.strip().removeprefix("```python").removeprefix("```").removesuffix("```").strip()
print("=== LLM-generirani kod ===")
print(kod)
print()
print("=== Rezultat ===")
exec(kod)

---
## 4. Embedding: semantička pretraga kulturnih zbirki

**Embedding** pretvara tekst u vektor brojeva; slični tekstovi = bliski vektori.
Ovo omogućuje **semantičku pretragu**: "morske teme u muzeju" nađe i zapise
koji ne sadrže riječ "more", ali govore o njemu (npr. "ribar", "brod", "val").

To je temelj modernih sustava preporuke i pretrage u digitalnoj humanistici.

In [ ]:
# @title Semantička pretraga zbirke
# Opisi djela iz zbirke
opisi = [
    "Portret starog ribara s mrežama na obali",
    "Mrtva priroda s cvijećem i voćem na stolu",
    "Pejzaž s jedrenjakom na pučini Kvarnera",
    "Apstraktna kompozicija u crvenoj i plavoj",
    "Stara fotografija riječke luke s parobrodom",
    "Skica djevojke s kišobranom u parku",
]

def embed(tekstovi):
    r = genai.embed_content(model="models/text-embedding-004", content=tekstovi)
    return np.array(r["embedding"])

vektori = embed(opisi)

from sklearn.metrics.pairwise import cosine_similarity

def semanticka_pretraga(upit, opisi, vektori, k=3):
    qv = embed([upit])
    sim = cosine_similarity(qv, vektori)[0]
    top = sim.argsort()[-k:][::-1]
    print(f"Upit: '{upit}'")
    for i in top:
        print(f"  [{sim[i]:.3f}] {opisi[i]}")
    print()

print("=== Semantička pretraga ===")
semanticka_pretraga("morski pejzaži i brodovi", opisi, vektori)
semanticka_pretraga("cvjetni motivi", opisi, vektori)

print("→ Riječ 'brod' se ne pojavljuje u svim rezultatima,")
print("  ali semantička blizina svejedno pronalazi relevantne zapise.")

---
## 5. RAG: razgovor s kulturnom baštinom

**RAG (Retrieval-Augmented Generation)** = dohvat + generiranje:
1. Korisnik postavi pitanje
2. Sustav pronađe najrelevantnije dijelove korpusa (embedding pretraga)
3. LLM generira odgovor **temeljen na dohvaćenim dokazima**

Rješava problem halucinacija: model odgovara iz **vašeg** korpusa, ne iz pamćenja.
Ovo je arhitektura današnjih AI asistenata za kulturnu baštinu (muzeji, arhivi).

In [ ]:
# @title Mini-RAG: pitaj zbirku dokumenata
korpus = [
    "Riječki port je sredinom 19. stoljeća postao jedna od najvažnijih luka Austro-Ugarske.",
    "Kazalište HNK Ivan pl. Zajc sagrađeno je 1885. godine u neorenesansnom stilu.",
    "Tornjačić na Trsatu potječe iz 13. stoljeća, a riječ je o najstarijem sačuvanom dijelu utvrde.",
    "Gradska vijećnica u Rijeci građena je od 1885. do 1894. prema projektu Janosa Wagnera.",
    "Prva riječka rafinerija nafte otvorena je 1882. godine, jedna od prvih u Europi.",
    "Palača Modello izgrađena je 1885. godine u historicističkom stilu.",
]

def rag_pitaj(pitanje, korpus, k=2):
    vk = embed(korpus)
    vp = embed([pitanje])
    sim = cosine_similarity(vp, vk)[0]
    top = sim.argsort()[-k:][::-1]
    kontekst = "\n\n".join(f"[{i+1}] {korpus[i]}" for i in top)

    prompt = f"""
Odgovori na pitanje ISKLJUČIVO na temelju priloženih dokaza.
Ako odgovor nije u dokazima, reci: "Nije navedeno u izvorima."
Citiraj izvor u zagradi: (izvor [N])

Dokazi:
{kontekst}

Pitanje: {pitanje}
"""
    odgovor = model.generate_content(prompt)
    print(f"Pitanje: {pitanje}")
    print(f"Dohvaćeni dokazi ({k}):")
    for i in top:
        print(f"  [{i+1}] {korpus[i][:70]}...")
    print(f"\nOdgovor: {odgovor.text}")
    print()

print("=== Mini-RAG demo ===")
rag_pitaj("Kada je sagrađeno riječko kazalište?", korpus)
rag_pitaj("Tko je projektirao Gradsku vijećnicu?", korpus)

---
## 6. Agentski sustavi: automatizirani istraživač

**Agent** = LLM + alati + petlja (razmisli → djeluj → promatraj → prilagodi).
U istraživanju kulture agent može: pretraživati katalog, računati statistiku,
generirati vizualizaciju i napisati izvješće — sve u jednom tijeku.

Knjiga *Komunikacija u doba umjetne inteligencije* ovaj oblik naziva
**komunikacijskim agentom**: sučelje koje razumije namjeru i izvršava ju kroz alate.

In [ ]:
# @title Mini agent: istraživač kulturne baštine (ReAct)
def pretrazi_katalog(upit: str) -> str:
    """Simulira dohvat iz muzejskog kataloga."""
    katalog = {
        "kazalište": ["HNK Zajc, 1885, neorenesansa"],
        "luka": ["Port, 19. st., Austro-Ugarska"],
        "toranj": ["Tornjačić Trsat, 13. st."],
        "rafinerija": ["Prva rafinerija, 1882"],
    }
    for k, v in katalog.items():
        if k in upit.lower():
            return ", ".join(v)
    return "Nema rezultata."

def agent_istrazivac(upit):
    print(f"1. REASON:  Trebam podatke o '{upit}'")
    rez = pretrazi_katalog(upit)
    print(f"2. ACT:     pretrazi_katalog('{upit}') → {rez}")
    print(f"3. OBSERVE: {rez}")
    print("4. ADJUST:  formuliram odgovor...")
    return f"Zaključak o '{upit}': {rez}"

print(agent_istrazivac("kazalište"))
print()
print("→ Produkcijski primjer (Google ADK 2.6):")
print("  LlmAgent(model='gemini-2.0-flash', tools=[FunctionTool(pretrazi_katalog)], instruction=...)")
print("  LoopAgent(sub_agents=[agent], max_iterations=5) — vidi scripts/06_agent_istrazivac.py")

---
## 7. Etika i kritičko vrednovanje AI rezultata

**Ključna vještina digitalne humanistike 2026.:** znati KADA vjerovati modelu.

| Rizik | Primjer u kulturi | Mitigacija |
|-------|-------------------|------------|
| Halucinacije | Izmišljeni podatak o autoru | RAG s dokazima, provjera izvora |
| Pristranost korpusa | Zbirke favoriziraju zapadne autore | Auditi reprezentacije |
| Anakronizmi | Moderni koncepti projicirani na prošlost | Kontekstualizacija, stručni pregled |
| Privatnost | Digitalizirani osobni podaci | Anonimizacija, etički okviri |

**Pravilo:** LLM je **asistent za prvi nacrt** — konačno tumačenje uvijek ostaje na istraživaču.

---
## 8. Vježbe

### 🟢 Osnovna
1. Učitaj vlastiti CSV (anketa/katalog) i generiraj 3 vizualizacije.
2. Pomoću LLM-a generiraj Pandas analizu iz prirodnojezičnog pitanja o svojim podacima.

### 🟡 Srednja
3. Napravi semantičku pretragu nad 10+ opisa kulturnih dobara; testiraj upite koji
   NISU doslovni (sinonimi, teme).
4. Izgradi mini-RAG nad svojim korpusom (10+ dokumenata) i testiraj odgovore s/bez konteksta.

### 🏆 Napredna
5. Kompletan AI pipeline: API prikupljanje → FAIR priprema → embedding → RAG → agent → izvješće.
6. U Google ADK 2.6 sagradi agenta s MCP alatom koji analizira kulturni sadržaj
   (npr. Wikipedia + katalog + vizualizacija) — dokumentiraj korake.

---
### 📖 Resursi i literatura
- **Perak, B. (2025). *Komunikacija u doba umjetne inteligencije*. FFRI.**
  [GitHub](https://github.com/bperak/komunikacija_u_doba_ai) | ISBN 978-953-361-147-1
- Grus, J. (2019). *Data Science from Scratch*. O'Reilly.
- Bommasani et al. (2021). *On the Opportunities and Risks of Foundation Models*. arXiv:2108.07258
- Python za lingviste: https://github.com/nljubesi/python-for-linguists
- Voyant Tools: https://voyant-tools.org | Recogito: https://recogito.pelagios.org
- Google AI Studio: https://aistudio.google.com | ADK: https://adk.dev

*Kolegij: Data Science u kulturi | 2026./2027.*